# Customer Segmentation Using K-Means
Group mall customers into meaningful segments based on **annual income** and **spending score**, so a business can target each group with the right marketing strategy.

**Steps:** Load data → EDA → Scale → Choose k (Elbow + Silhouette) → K-Means → Profile & name segments → Business recommendations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid", palette="deep")
%matplotlib inline

## 1. Load the data

In [ ]:
df = pd.read_csv("../data/Mall_Customers.csv")
df.head()

In [ ]:
print(df.shape)
df.info()
print("Missing values:", df.isna().sum().sum())
print("Duplicates:", df.duplicated().sum())
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, features):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

In [ ]:
print(df["Gender"].value_counts())
df.groupby("Gender")[features].mean().round(1)

In [ ]:
sns.heatmap(df[features].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 3. Feature selection and scaling
K-Means uses distances, so features must be on the same scale. We cluster on **Annual Income** and **Spending Score**; Age is kept for profiling the segments afterwards.

In [ ]:
cluster_features = ["Annual Income (k$)", "Spending Score (1-100)"]
X = df[cluster_features]
scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)

## 4. Choose the number of clusters (k)

In [ ]:
k_range = range(2, 11)
inertias, sil_scores = [], []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set(title="Elbow Method", xlabel="k", ylabel="Inertia (WCSS)")
axes[1].plot(list(k_range), sil_scores, marker="o", color="green")
axes[1].set(title="Silhouette Score", xlabel="k", ylabel="Score")
plt.tight_layout()
plt.show()

best_k = list(k_range)[int(np.argmax(sil_scores))]
print("Best k:", best_k, "| silhouette =", round(max(sil_scores), 3))

## 5. Train the final K-Means model

In [ ]:
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X_scaled)
df["Cluster"] = kmeans.labels_
df["Cluster"].value_counts().sort_index()

## 6. Profile and name the segments

In [ ]:
profile = df.groupby("Cluster")[features].mean().round(1)
profile["Customers"] = df["Cluster"].value_counts().sort_index()

def name_segment(row):
    inc, spd = row["Annual Income (k$)"], row["Spending Score (1-100)"]
    inc_lo, inc_hi = df["Annual Income (k$)"].quantile([0.33, 0.67])
    spd_lo, spd_hi = df["Spending Score (1-100)"].quantile([0.33, 0.67])
    inc_lvl = "Low" if inc < inc_lo else ("High" if inc > inc_hi else "Mid")
    spd_lvl = "Low" if spd < spd_lo else ("High" if spd > spd_hi else "Mid")
    if inc_lvl == "Mid" and spd_lvl == "Mid": return "Standard (Average Income, Average Spend)"
    if inc_lvl == "High" and spd_lvl == "High": return "Premium (High Income, High Spend)"
    if inc_lvl == "High": return "Careful (High Income, Low Spend)"
    if spd_lvl == "High": return "Impulsive (Low Income, High Spend)"
    return "Budget (Low Income, Low Spend)"

profile["Segment"] = profile.apply(name_segment, axis=1)
df["Segment"] = df["Cluster"].map(profile["Segment"])
profile

## 7. Visualise the segments

In [ ]:
centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=cluster_features)
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)", hue="Segment", s=70)
plt.scatter(centers.iloc[:, 0], centers.iloc[:, 1], c="black", marker="X", s=250, label="Centroids")
plt.title("Customer Segments")
plt.legend(fontsize=8)
plt.show()

In [ ]:
pcs = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=df["Segment"], s=70)
plt.title("Clusters in 2D (PCA)")
plt.show()

In [ ]:
print(pd.crosstab(df["Segment"], df["Gender"], normalize="index").round(2))

## 8. Business recommendations
| Segment | Strategy |
|---|---|
| **Premium** (high income, high spend) | VIP loyalty program, early access to new products, premium services |
| **Careful** (high income, low spend) | Personalised offers and reminders; find out what would make them spend more |
| **Impulsive** (low income, high spend) | Flash sales, limited-time deals, bundle offers, EMI options |
| **Standard** (average income, average spend) | Regular promotions, upsell and cross-sell |
| **Budget** (low income, low spend) | Discount coupons and low-cost entry products; low marketing spend |

In [ ]:
df.to_csv("../results/customers_with_clusters.csv", index=False)
profile.to_csv("../results/cluster_profile.csv")
print("Results saved.")

## 9. Conclusion
K-Means with k=5 (chosen with the Elbow method and Silhouette score) separates customers into five clear groups. The segments give the business a data-driven way to target marketing, loyalty and pricing decisions.

**Limitations / future work:** only 2 features used for clustering; try other algorithms (DBSCAN, hierarchical, Gaussian Mixture), add more behavioural features (purchase frequency, recency), and build a Streamlit app to predict a new customer's segment.